In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
df_github = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/GitHub_bugs.csv')
print(df_github.shape)
print(df_github.columns.tolist())
print(df_github.head())

Mounted at /content/drive
(32829, 6)
['Issue id', 'Summary', 'Created', 'Resolved', 'Description', 'Resolution']
   Issue id                                            Summary  \
0    223706  Configure unassigned keybindings, command name...   
1    223658  Some suggestions are missing on Windows with pwsh   
2    223641  Cannot split a terminal without a group [objec...   
3    223622  SCM - history graph handling of first commit i...   
4    223607          SCM - history graph using incorrect color   

                     Created                   Resolved  \
0  2024-07-25 14:34:59+00:00                        NaN   
1  2024-07-25 13:06:40+00:00  2024-07-25 16:14:57+00:00   
2  2024-07-25 12:19:09+00:00                        NaN   
3  2024-07-25 10:44:47+00:00                        NaN   
4  2024-07-25 09:43:02+00:00  2024-07-25 11:02:46+00:00   

                                         Description Resolution  
0  - Place cursor in the panel chat\r\n- Open the...        NaN  
1  

In [ ]:
import pandas as pd
from sklearn.utils import resample

# Step 1 - Load
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/GitHub_bugs.csv')

# Step 2 - Text combine
df_github = df.copy()
df_github['text'] = df_github['Summary'].fillna('') + ' ' + df_github['Description'].fillna('')

# Step 3 - Severity labeling
def assign_severity(text):
    text = str(text).lower()

    if any(word in text for word in ['urgent', 'immediately', 'asap', 'critical',
                                      'emergency', 'crash', 'broken', 'blocked',
                                      'cannot', 'unable', 'failed', 'error']):
        return 'Critical'
    elif any(word in text for word in ['frustrated', 'frustrating', 'wrong',
                                        'incorrect', 'issue', 'problem',
                                        'anxious', 'worried', 'not working']):
        return 'High'
    else:
        return 'Low'

df_github['severity'] = df_github['text'].apply(assign_severity)

# Step 4 - Balance
total = len(df_github)
n_samples = total // 3
print(f"Total: {total}, Each class: {n_samples}")
print("\nBefore balance:")
print(df_github['severity'].value_counts())

df_critical = df_github[df_github['severity'] == 'Critical']
df_high = df_github[df_github['severity'] == 'High']
df_low = df_github[df_github['severity'] == 'Low']

df_critical_bal = resample(df_critical, replace=True, n_samples=n_samples, random_state=42)
df_high_bal = resample(df_high, replace=True, n_samples=n_samples, random_state=42)
df_low_bal = resample(df_low, replace=True, n_samples=n_samples, random_state=42)

df_balanced = pd.concat([df_critical_bal, df_high_bal, df_low_bal])
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print("\nAfter balance:")
print(df_balanced['severity'].value_counts())
print(f"Total: {len(df_balanced)}")

# Step 5 - Save
df_balanced.to_csv('/content/drive/MyDrive/Colab Notebooks/github_severity_balanced.csv', index=False)
print("\nSaved!")

Total: 32829, Each class: 10943

Before balance:
severity
Low         12819
High        10828
Critical     9182
Name: count, dtype: int64

After balance:
severity
Low         10943
High        10943
Critical    10943
Name: count, dtype: int64
Total: 32829

Saved!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

# Load balanced dataset
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/github_severity_balanced.csv')
print(df.shape)
print(df.columns.tolist())
print(df['severity'].value_counts())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(32829, 8)
['Issue id', 'Summary', 'Created', 'Resolved', 'Description', 'Resolution', 'text', 'severity']
severity
Low         10943
High        10943
Critical    10943
Name: count, dtype: int64


In [ ]:
import re

# CLEAN FUNCTION
def clean_text(text):

    text = str(text).lower()

    text = re.sub(r"http\S+", "", text)

    text = re.sub(r"<.*?>", "", text)

    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()

# CREATE CLEAN TEXT
df["clean_text"] = df["text"].apply(clean_text)

print(
    df[["text", "clean_text"]].head()
)

                                                text  \
0  Explorer rename text field screen cheese Notic...   
1  Settings: bad overflow behaviour of setting ta...   
2  No textDocument/signatureHelp message is sent ...   
3  Notifications no longer detect duplicates if r...   
4  Variable selection starting with dollar sign -...   

                                          clean_text  
0  explorer rename text field screen cheese notic...  
1  settings bad overflow behaviour of setting tab...  
2  no textdocument signaturehelp message is sent ...  
3  notifications no longer detect duplicates if r...  
4  variable selection starting with dollar sign v...  


Train/Test Split + TF-IDF

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib

# SPLIT
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["severity"]
)

# TEXT + LABELS
X_train_text = train_df["clean_text"]

X_test_text = test_df["clean_text"]

y_train = train_df["severity"]

y_test = test_df["severity"]

# TF-IDF
github_vectorizer = TfidfVectorizer(
    max_features=10000,
    stop_words="english"
)

X_train = github_vectorizer.fit_transform(
    X_train_text
)

X_test = github_vectorizer.transform(
    X_test_text
)

print(X_train.shape)
print(X_test.shape)

# SAVE
joblib.dump(
    github_vectorizer,
    "/content/drive/MyDrive/Colab Notebooks/github_severity_tfidf_vectorizer.pkl"
)

print("GitHub Severity TF-IDF Saved")

(26263, 10000)
(6566, 10000)
GitHub Severity TF-IDF Saved


Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import joblib

# TRAIN MODEL
github_lr_model = LogisticRegression(
    max_iter=1000
)

github_lr_model.fit(
    X_train,
    y_train
)

print("GitHub Severity Logistic Regression Model Trained")

# PREDICT
y_pred = github_lr_model.predict(
    X_test
)

# ACCURACY
github_lr_acc = accuracy_score(
    y_test,
    y_pred
)

print("Accuracy:", github_lr_acc)

# REPORT
print("\nClassification Report:\n")

print(classification_report(
    y_test,
    y_pred
))

# SAVE MODEL
joblib.dump(
    github_lr_model,
    "/content/drive/MyDrive/Colab Notebooks/github_severity_logistic_regression_model.pkl"
)

print("GitHub Severity Logistic Regression Model Saved")

GitHub Severity Logistic Regression Model Trained
Accuracy: 0.8827292110874201

Classification Report:

              precision    recall  f1-score   support

    Critical       0.95      0.86      0.90      2188
        High       0.89      0.82      0.85      2189
         Low       0.83      0.97      0.89      2189

    accuracy                           0.88      6566
   macro avg       0.89      0.88      0.88      6566
weighted avg       0.89      0.88      0.88      6566

GitHub Severity Logistic Regression Model Saved


Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

# TRAIN MODEL
github_rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

github_rf_model.fit(
    X_train,
    y_train
)

print("GitHub Severity Random Forest Model Trained")

# PREDICT
y_pred = github_rf_model.predict(
    X_test
)

# ACCURACY
github_rf_acc = accuracy_score(
    y_test,
    y_pred
)

print("Accuracy:", github_rf_acc)

# REPORT
print("\nClassification Report:\n")

print(classification_report(
    y_test,
    y_pred
))

# SAVE MODEL
joblib.dump(
    github_rf_model,
    "/content/drive/MyDrive/Colab Notebooks/github_severity_random_forest_model.pkl"
)

print("GitHub Severity Random Forest Model Saved")

GitHub Severity Random Forest Model Trained
Accuracy: 0.9456289978678039

Classification Report:

              precision    recall  f1-score   support

    Critical       0.98      0.93      0.95      2188
        High       0.94      0.92      0.93      2189
         Low       0.92      0.99      0.96      2189

    accuracy                           0.95      6566
   macro avg       0.95      0.95      0.95      6566
weighted avg       0.95      0.95      0.95      6566

GitHub Severity Random Forest Model Saved


SVM

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report
import joblib

# TRAIN MODEL
github_svm_model = LinearSVC()

github_svm_model.fit(
    X_train,
    y_train
)

print("GitHub Severity SVM Model Trained")

# PREDICT
y_pred = github_svm_model.predict(
    X_test
)

# ACCURACY
github_svm_acc = accuracy_score(
    y_test,
    y_pred
)

print("Accuracy:", github_svm_acc)

# REPORT
print("\nClassification Report:\n")

print(classification_report(
    y_test,
    y_pred
))

# SAVE MODEL
joblib.dump(
    github_svm_model,
    "/content/drive/MyDrive/Colab Notebooks/github_severity_svm_model.pkl"
)

print("GitHub Severity SVM Model Saved")

GitHub Severity SVM Model Trained
Accuracy: 0.9107523606457508

Classification Report:

              precision    recall  f1-score   support

    Critical       0.97      0.91      0.94      2188
        High       0.91      0.87      0.89      2189
         Low       0.86      0.96      0.91      2189

    accuracy                           0.91      6566
   macro avg       0.91      0.91      0.91      6566
weighted avg       0.91      0.91      0.91      6566

GitHub Severity SVM Model Saved


Naive Bayes

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
import joblib

# TRAIN MODEL
github_nb_model = MultinomialNB()

github_nb_model.fit(
    X_train,
    y_train
)

print("GitHub Severity Naive Bayes Model Trained")

# PREDICT
y_pred = github_nb_model.predict(
    X_test
)

# ACCURACY
github_nb_acc = accuracy_score(
    y_test,
    y_pred
)

print("Accuracy:", github_nb_acc)

# REPORT
print("\nClassification Report:\n")

print(classification_report(
    y_test,
    y_pred
))

# SAVE MODEL
joblib.dump(
    github_nb_model,
    "/content/drive/MyDrive/Colab Notebooks/github_severity_naive_bayes_model.pkl"
)

print("GitHub Severity Naive Bayes Model Saved")

GitHub Severity Naive Bayes Model Trained
Accuracy: 0.7322570819372525

Classification Report:

              precision    recall  f1-score   support

    Critical       0.83      0.67      0.74      2188
        High       0.71      0.69      0.70      2189
         Low       0.69      0.84      0.75      2189

    accuracy                           0.73      6566
   macro avg       0.74      0.73      0.73      6566
weighted avg       0.74      0.73      0.73      6566

GitHub Severity Naive Bayes Model Saved


XGBoost

In [ ]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import joblib

# LABEL ENCODING
severity_encoder = LabelEncoder()

y_train_encoded = severity_encoder.fit_transform(y_train)

y_test_encoded = severity_encoder.transform(y_test)

# TRAIN MODEL
github_xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    objective='multi:softmax',
    num_class=3,
    random_state=42
)

github_xgb_model.fit(
    X_train,
    y_train_encoded
)

print("GitHub Severity XGBoost Model Trained")

# PREDICT
y_pred_encoded = github_xgb_model.predict(
    X_test
)

# DECODE LABELS
y_pred = severity_encoder.inverse_transform(
    y_pred_encoded
)

# ACCURACY
github_xgb_acc = accuracy_score(
    y_test,
    y_pred
)

print("Accuracy:", github_xgb_acc)

# REPORT
print("\nClassification Report:\n")

print(classification_report(
    y_test,
    y_pred
))

# SAVE MODEL
joblib.dump(
    github_xgb_model,
    "/content/drive/MyDrive/Colab Notebooks/github_severity_xgboost_model.pkl"
)

print("GitHub Severity XGBoost Model Saved")

GitHub Severity XGBoost Model Trained
Accuracy: 0.9218702406335668

Classification Report:

              precision    recall  f1-score   support

    Critical       1.00      0.89      0.94      2188
        High       0.93      0.89      0.91      2189
         Low       0.85      0.99      0.92      2189

    accuracy                           0.92      6566
   macro avg       0.93      0.92      0.92      6566
weighted avg       0.93      0.92      0.92      6566

GitHub Severity XGBoost Model Saved


LSTM

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

import numpy as np
import pickle

# TOKENIZATION
tokenizer = Tokenizer(num_words=10000)

tokenizer.fit_on_texts(
    train_df["clean_text"]
)

X_train_seq = tokenizer.texts_to_sequences(
    train_df["clean_text"]
)

X_test_seq = tokenizer.texts_to_sequences(
    test_df["clean_text"]
)

# PAD SEQUENCES
max_len = 100

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=max_len
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=max_len
)

# LABEL ENCODING
label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(
    y_train
)

y_test_encoded = label_encoder.transform(
    y_test
)

# BUILD MODEL
github_lstm_model = Sequential()

github_lstm_model.add(
    Embedding(
        input_dim=10000,
        output_dim=128,
        input_length=max_len
    )
)

github_lstm_model.add(
    LSTM(128)
)

github_lstm_model.add(
    Dropout(0.5)
)

github_lstm_model.add(
    Dense(64, activation='relu')
)

github_lstm_model.add(
    Dense(3, activation='softmax')
)

# COMPILE
github_lstm_model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

# TRAIN
github_lstm_model.fit(
    X_train_pad,
    y_train_encoded,
    epochs=5,
    batch_size=32,
    validation_split=0.1
)

# EVALUATE
loss, accuracy = github_lstm_model.evaluate(
    X_test_pad,
    y_test_encoded
)

print("GitHub Severity LSTM Accuracy:", accuracy)

# PREDICT
y_pred_probs = github_lstm_model.predict(
    X_test_pad
)

y_pred = np.argmax(
    y_pred_probs,
    axis=1
)

# REPORT
print("\nClassification Report:\n")

print(classification_report(
    y_test_encoded,
    y_pred,
    target_names=label_encoder.classes_
))

# SAVE MODEL
github_lstm_model.save(
    "/content/drive/MyDrive/Colab Notebooks/github_severity_lstm_model.keras"
)

# SAVE TOKENIZER
with open(
    "/content/drive/MyDrive/Colab Notebooks/github_severity_tokenizer.pkl",
    "wb"
) as f:

    pickle.dump(tokenizer, f)

print("GitHub Severity LSTM Model Saved")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/5
739/739 ━━━━━━━━━━━━━━━━━━━━ 150s 198ms/step - accuracy: 0.7453 - loss: 0.6488 - val_accuracy: 0.8652 - val_loss: 0.4300
Epoch 2/5
739/739 ━━━━━━━━━━━━━━━━━━━━ 149s 201ms/step - accuracy: 0.8773 - loss: 0.3759 - val_accuracy: 0.8900 - val_loss: 0.3413
Epoch 3/5
739/739 ━━━━━━━━━━━━━━━━━━━━ 150s 203ms/step - accuracy: 0.8999 - loss: 0.3133 - val_accuracy: 0.8904 - val_loss: 0.3153
Epoch 4/5
739/739 ━━━━━━━━━━━━━━━━━━━━ 201s 202ms/step - accuracy: 0.9024 - loss: 0.2814 - val_accuracy: 0.8976 - val_loss: 0.3036
Epoch 5/5
739/739 ━━━━━━━━━━━━━━━━━━━━ 154s 209ms/step - accuracy: 0.9277 - loss: 0.2083 - val_accuracy: 0.8820 - val_loss: 0.3036
206/206 ━━━━━━━━━━━━━━━━━━━━ 13s 62ms/step - accuracy: 0.8754 - loss: 0.3255
GitHub Severity LSTM Accuracy: 0.8754188418388367
206/206 ━━━━━━━━━━━━━━━━━━━━ 14s 65ms/step

Classification Report:

              precision    recall  f1-score   support

    Critical       0.95      0.91      0.93      2188
        High       0.78      0.89      0.

BiLSTM

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional
from tensorflow.keras.layers import LSTM, Dense, Dropout

from sklearn.metrics import classification_report

import numpy as np

# BUILD MODEL
github_bilstm_model = Sequential()

github_bilstm_model.add(
    Embedding(
        input_dim=10000,
        output_dim=128,
        input_length=max_len
    )
)

github_bilstm_model.add(
    Bidirectional(
        LSTM(128)
    )
)

github_bilstm_model.add(
    Dropout(0.5)
)

github_bilstm_model.add(
    Dense(64, activation='relu')
)

github_bilstm_model.add(
    Dense(3, activation='softmax')
)

# COMPILE
github_bilstm_model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

# TRAIN
github_bilstm_model.fit(
    X_train_pad,
    y_train_encoded,
    epochs=5,
    batch_size=32,
    validation_split=0.1
)

# EVALUATE
loss, accuracy = github_bilstm_model.evaluate(
    X_test_pad,
    y_test_encoded
)

print("GitHub Severity BiLSTM Accuracy:", accuracy)

# PREDICT
y_pred_probs = github_bilstm_model.predict(
    X_test_pad
)

y_pred = np.argmax(
    y_pred_probs,
    axis=1
)

# REPORT
print("\nClassification Report:\n")

print(classification_report(
    y_test_encoded,
    y_pred,
    target_names=label_encoder.classes_
))

# SAVE MODEL
github_bilstm_model.save(
    "/content/drive/MyDrive/Colab Notebooks/github_severity_bilstm_model.keras"
)

print("GitHub Severity BiLSTM Model Saved")

Epoch 1/5


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


739/739 ━━━━━━━━━━━━━━━━━━━━ 292s 383ms/step - accuracy: 0.7612 - loss: 0.5798 - val_accuracy: 0.8938 - val_loss: 0.2963
Epoch 2/5
739/739 ━━━━━━━━━━━━━━━━━━━━ 282s 381ms/step - accuracy: 0.8968 - loss: 0.2938 - val_accuracy: 0.9033 - val_loss: 0.2763
Epoch 3/5
739/739 ━━━━━━━━━━━━━━━━━━━━ 278s 376ms/step - accuracy: 0.9282 - loss: 0.2039 - val_accuracy: 0.9223 - val_loss: 0.2301
Epoch 4/5
739/739 ━━━━━━━━━━━━━━━━━━━━ 277s 375ms/step - accuracy: 0.9495 - loss: 0.1500 - val_accuracy: 0.9223 - val_loss: 0.2528
Epoch 5/5
739/739 ━━━━━━━━━━━━━━━━━━━━ 273s 369ms/step - accuracy: 0.9547 - loss: 0.1296 - val_accuracy: 0.9303 - val_loss: 0.2306
206/206 ━━━━━━━━━━━━━━━━━━━━ 19s 94ms/step - accuracy: 0.9098 - loss: 0.3051
GitHub Severity BiLSTM Accuracy: 0.9098385572433472
206/206 ━━━━━━━━━━━━━━━━━━━━ 21s 100ms/step

Classification Report:

              precision    recall  f1-score   support

    Critical       0.93      0.94      0.94      2188
        High       0.89      0.85      0.87     

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


CNN + BiLSTM

In [ ]:
# ==========================================
# DRIVE MOUNT
# ==========================================

from google.colab import drive
drive.mount('/content/drive')

# ==========================================
# IMPORT LIBRARIES
# ==========================================

import pandas as pd
import re
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, MaxPooling1D
from tensorflow.keras.layers import Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# ==========================================
# LOAD DATASET
# ==========================================

df = pd.read_csv(
    "/content/drive/MyDrive/Colab Notebooks/github_severity_balanced.csv"
)

print(df.shape)
print(df["severity"].value_counts())

# ==========================================
# CLEAN TEXT
# ==========================================

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["clean_text"] = df["text"].apply(clean_text)

# ==========================================
# TRAIN TEST SPLIT
# ==========================================

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["severity"]
)

# ==========================================
# TOKENIZATION
# ==========================================

MAX_WORDS = 10000
MAX_LEN = 100

tokenizer = Tokenizer(num_words=MAX_WORDS)

tokenizer.fit_on_texts(train_df["clean_text"])

X_train_seq = tokenizer.texts_to_sequences(train_df["clean_text"])
X_test_seq = tokenizer.texts_to_sequences(test_df["clean_text"])

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN)
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN)

# ==========================================
# LABEL ENCODING
# ==========================================

label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(train_df["severity"])
y_test_encoded = label_encoder.transform(test_df["severity"])

print(label_encoder.classes_)

# ==========================================
# CNN + BiLSTM MODEL
# ==========================================

github_cnn_bilstm_model = Sequential([
    Embedding(MAX_WORDS, 128, input_length=MAX_LEN),

    Conv1D(
        filters=128,
        kernel_size=5,
        activation="relu"
    ),

    MaxPooling1D(pool_size=2),

    Bidirectional(
        LSTM(128)
    ),

    Dropout(0.5),

    Dense(64, activation="relu"),

    Dense(3, activation="softmax")
])

github_cnn_bilstm_model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

# ==========================================
# TRAIN MODEL
# ==========================================

history = github_cnn_bilstm_model.fit(
    X_train_pad,
    y_train_encoded,
    epochs=5,
    batch_size=32,
    validation_split=0.1
)

# ==========================================
# EVALUATE MODEL
# ==========================================

loss, accuracy = github_cnn_bilstm_model.evaluate(
    X_test_pad,
    y_test_encoded
)

print("GitHub Severity CNN + BiLSTM Accuracy:", accuracy)

# ==========================================
# CLASSIFICATION REPORT
# ==========================================

y_pred_probs = github_cnn_bilstm_model.predict(X_test_pad)

y_pred = np.argmax(y_pred_probs, axis=1)

print("\nClassification Report:\n")

print(classification_report(
    y_test_encoded,
    y_pred,
    target_names=label_encoder.classes_
))

# ==========================================
# SAVE MODEL + TOKENIZER
# ==========================================

github_cnn_bilstm_model.save(
    "/content/drive/MyDrive/Colab Notebooks/github_severity_cnn_bilstm_model.keras"
)

with open(
    "/content/drive/MyDrive/Colab Notebooks/github_severity_tokenizer.pkl",
    "wb"
) as f:
    pickle.dump(tokenizer, f)

print("GitHub Severity CNN + BiLSTM Model Saved")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(32829, 8)
severity
Low         10943
High        10943
Critical    10943
Name: count, dtype: int64
['Critical' 'High' 'Low']


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/5
739/739 ━━━━━━━━━━━━━━━━━━━━ 18s 12ms/step - accuracy: 0.8034 - loss: 0.5036 - val_accuracy: 0.8953 - val_loss: 0.2822
Epoch 2/5
739/739 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.9136 - loss: 0.2486 - val_accuracy: 0.9201 - val_loss: 0.2346
Epoch 3/5
739/739 ━━━━━━━━━━━━━━━━━━━━ 10s 13ms/step - accuracy: 0.9411 - loss: 0.1764 - val_accuracy: 0.9204 - val_loss: 0.2132
Epoch 4/5
739/739 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.9608 - loss: 0.1183 - val_accuracy: 0.9357 - val_loss: 0.2244
Epoch 5/5
739/739 ━━━━━━━━━━━━━━━━━━━━ 10s 13ms/step - accuracy: 0.9757 - loss: 0.0734 - val_accuracy: 0.9353 - val_loss: 0.2345
206/206 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9243 - loss: 0.2859
GitHub Severity CNN + BiLSTM Accuracy: 0.9243070483207703
206/206 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step

Classification Report:

              precision    recall  f1-score   support

    Critical       0.95      0.94      0.95      2188
        High       0.90      0.88      0.89      

BERT

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import TrainingArguments, Trainer

import torch
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

# LABEL ENCODING
label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(train_df["severity"])
y_test_encoded = label_encoder.transform(test_df["severity"])

# TOKENIZER
bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

train_encodings = bert_tokenizer(
    train_df["clean_text"].tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = bert_tokenizer(
    test_df["clean_text"].tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

class SeverityDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = SeverityDataset(train_encodings, y_train_encoded)
test_dataset = SeverityDataset(test_encodings, y_test_encoded)

github_bert_model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3
)

training_args = TrainingArguments(
    output_dir="./github_bert_results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    save_strategy="no"
)

github_bert_trainer = Trainer(
    model=github_bert_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

github_bert_trainer.train()

predictions = github_bert_trainer.predict(test_dataset)

y_pred = np.argmax(predictions.predictions, axis=1)

github_bert_acc = accuracy_score(y_test_encoded, y_pred)

print("GitHub Severity BERT Accuracy:", github_bert_acc)

print(classification_report(
    y_test_encoded,
    y_pred,
    target_names=label_encoder.classes_
))

github_bert_model.save_pretrained(
    "/content/drive/MyDrive/Colab Notebooks/github_severity_bert_model"
)

bert_tokenizer.save_pretrained(
    "/content/drive/MyDrive/Colab Notebooks/github_severity_bert_model"
)

print("GitHub Severity BERT Model Saved")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

... (2 outputs left out)

Step,Training Loss
500,0.565806
1000,0.350691
1500,0.306396
2000,0.265271
2500,0.229387
3000,0.201512
3500,0.157518
4000,0.123521
4500,0.133583


GitHub Severity BERT Accuracy: 0.9390801096558026
              precision    recall  f1-score   support

    Critical       0.96      0.96      0.96      2188
        High       0.93      0.90      0.91      2189
         Low       0.93      0.96      0.95      2189

    accuracy                           0.94      6566
   macro avg       0.94      0.94      0.94      6566
weighted avg       0.94      0.94      0.94      6566



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

GitHub Severity BERT Model Saved


RoBERTa

In [ ]:
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from transformers import TrainingArguments, Trainer

import torch
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

# LABEL ENCODING
label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(train_df["severity"])
y_test_encoded = label_encoder.transform(test_df["severity"])

# TOKENIZER
roberta_tokenizer = RobertaTokenizer.from_pretrained(
    "roberta-base"
)

train_encodings = roberta_tokenizer(
    train_df["clean_text"].tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = roberta_tokenizer(
    test_df["clean_text"].tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

class SeverityDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(self.labels[idx])

        return item

    def __len__(self):
        return len(self.labels)

train_dataset = SeverityDataset(train_encodings, y_train_encoded)
test_dataset = SeverityDataset(test_encodings, y_test_encoded)

# MODEL
github_roberta_model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=3
)

training_args = TrainingArguments(
    output_dir="./github_roberta_results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    save_strategy="no"
)

github_roberta_trainer = Trainer(
    model=github_roberta_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

# TRAIN
github_roberta_trainer.train()

# PREDICT
predictions = github_roberta_trainer.predict(test_dataset)

y_pred = np.argmax(predictions.predictions, axis=1)

# ACCURACY
github_roberta_acc = accuracy_score(
    y_test_encoded,
    y_pred
)

print("GitHub Severity RoBERTa Accuracy:", github_roberta_acc)

print(classification_report(
    y_test_encoded,
    y_pred,
    target_names=label_encoder.classes_
))

# SAVE MODEL
github_roberta_model.save_pretrained(
    "/content/drive/MyDrive/Colab Notebooks/github_severity_roberta_model"
)

roberta_tokenizer.save_pretrained(
    "/content/drive/MyDrive/Colab Notebooks/github_severity_roberta_model"
)

print("GitHub Severity RoBERTa Model Saved")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

... (2 outputs left out)

Step,Training Loss
500,0.515992
1000,0.349851
1500,0.309224
2000,0.275748
2500,0.235584
3000,0.216633
3500,0.198404
4000,0.168730
4500,0.174900


GitHub Severity RoBERTa Accuracy: 0.931160523911057
              precision    recall  f1-score   support

    Critical       0.95      0.96      0.95      2188
        High       0.93      0.87      0.90      2189
         Low       0.92      0.96      0.94      2189

    accuracy                           0.93      6566
   macro avg       0.93      0.93      0.93      6566
weighted avg       0.93      0.93      0.93      6566



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

GitHub Severity RoBERTa Model Saved


CodeBERT

In [7]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer

import torch
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

# LABEL ENCODING
label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(
    train_df["severity"]
)

y_test_encoded = label_encoder.transform(
    test_df["severity"]
)

# TOKENIZER
codebert_tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/codebert-base"
)

# TOKENIZATION
train_encodings = codebert_tokenizer(
    train_df["clean_text"].tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = codebert_tokenizer(
    test_df["clean_text"].tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

# DATASET CLASS
class SeverityDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(
            self.labels[idx]
        )

        return item

    def __len__(self):
        return len(self.labels)

# DATASETS
train_dataset = SeverityDataset(
    train_encodings,
    y_train_encoded
)

test_dataset = SeverityDataset(
    test_encodings,
    y_test_encoded
)

# MODEL
github_codebert_model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/codebert-base",
    num_labels=3
)

# TRAINING ARGUMENTS
training_args = TrainingArguments(
    output_dir="./github_codebert_results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    save_strategy="no"
)

# TRAINER
github_codebert_trainer = Trainer(
    model=github_codebert_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

# TRAIN
github_codebert_trainer.train()

# PREDICT
predictions = github_codebert_trainer.predict(
    test_dataset
)

y_pred = np.argmax(
    predictions.predictions,
    axis=1
)

# ACCURACY
github_codebert_acc = accuracy_score(
    y_test_encoded,
    y_pred
)

print("GitHub Severity CodeBERT Accuracy:", github_codebert_acc)

# REPORT
print("\nClassification Report:\n")

print(classification_report(
    y_test_encoded,
    y_pred,
    target_names=label_encoder.classes_
))

# SAVE MODEL
github_codebert_model.save_pretrained(
    "/content/drive/MyDrive/Colab Notebooks/github_severity_codebert_model"
)

codebert_tokenizer.save_pretrained(
    "/content/drive/MyDrive/Colab Notebooks/github_severity_codebert_model"
)

print("GitHub Severity CodeBERT Model Saved")

config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

... (4 outputs left out)

Step,Training Loss
500,0.560357
1000,0.338717
1500,0.300683
2000,0.242879
2500,0.219591
3000,0.200234
3500,0.178974
4000,0.152087
4500,0.154557


GitHub Severity CodeBERT Accuracy: 0.9361864148644532

Classification Report:

              precision    recall  f1-score   support

    Critical       0.95      0.96      0.95      2188
        High       0.94      0.88      0.91      2189
         Low       0.92      0.97      0.94      2189

    accuracy                           0.94      6566
   macro avg       0.94      0.94      0.94      6566
weighted avg       0.94      0.94      0.94      6566



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

GitHub Severity CodeBERT Model Saved
